In [1]:
import pandas as pd
from transformers import T5Tokenizer , Trainer ,T5ForConditionalGeneration ,TrainingArguments

C:\Users\prith\anaconda3\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data =pd.read_csv("dataset/samsum-train.csv")
val_data = pd.read_csv("dataset/samsum-validation.csv")

In [3]:
train_data.sample(5)

,id,dialogue,summary
9172,13716810,Anita: Are you in Warsaw already?\r\nMelissa: ...,Anita will pick Melissa and Kelly up from the ...
2059,13821224,Abigail: <file_video>\r\nAbigail: Daily portio...,Abigail tries to cheer up Olivia and Kate with...
9114,13730946,"Leo: Hi J, how is it going?\r\nJames: not so b...",James has been there for 2 weeks and it's stil...
10835,13815889,Kylie: I have heard that you were with Alex La...,Eli was with Alex last night. His parents were...
4803,13728042,Adam: need a drink\r\nAdam: wanna go to Paddys...,Adam and Will are going to Paddys at 7.


In [4]:
train_data.shape

(14732, 3)

In [5]:
# ramdom sampling
train_data = train_data.sample(n = 4000 , random_state =42).reset_index(drop=True)
val_data = val_data.sample(n=500 ,random_state =42).reset_index(drop=True)

In [6]:
train_data.shape

(4000, 3)

# data preprocessing

In [7]:
import re # regex

def  clean_data(text):
    text = re.sub(r"\r\n"," ",text) # lines
    text = re.sub(r"\s+" ," ",text) # extra spaces
    text = re.sub(r"<.*?>"," ", text) #html
    text = text.strip().lower()
    return text

In [8]:
train_data["dialogue"] =train_data["dialogue"].apply(clean_data)
train_data["summary"] =train_data["summary"].apply(clean_data)

val_data["dialogue"] =val_data["dialogue"].apply(clean_data)
val_data["summary"] =val_data["summary"].apply(clean_data)

# Tokenize

In [9]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [12]:
def tokenize(data):
    inputs = tokenizer(data["dialogue"] ,padding="max_length" ,max_length=512 ,truncation = True)
    targets = tokenizer(data["summary"] , padding="max_length" , max_length=150,truncation = True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize , axis=1).tolist()
val_dataset = val_data.apply(tokenize , axis=1).tolist()